In [1]:
import cv2

import numpy as np
from PIL import Image
from IPython import display



def load(fn):
    d=open(fn).read().split('\n\n')
    
    p={}
    for x in d[0:-1]:
        x = x.strip().split('\n')
        ix = int(x[0][:-1])
        pp = []
        w=0
        h=0
        for y, l in enumerate(x[1:]):
            h = max(y+1, h)
            for x, v in enumerate(l):
                w = max(x+1, w)
                if v == '#':
                    pp.append((x,y))
        p[ix]=pp
    
    s=[]
    for x in d[-1].split('\n'):
        x=x.split(' ')
        #print(x)
        w,h=[int(y) for y in x[0][:-1].split('x')]
        ix=[int(y) for y in x[1:]]
        #print(ix)
        _ix=[]
        for i,v in enumerate(ix):
            for x in range(v):
                _ix.append(i)
        #print(_ix)
        ix=_ix
        s.append((w,h,ix))
        #print(s[-1])
    
    
    return s,p

def home(p):
    #print(p)
    xx=min([x for x,y in p])
    yy=min([y for x,y in p])
    return [(x-xx,y-yy) for x,y in p]

def sz(p):
    return (max([x for x,y in p])+1,max([y for x,y in p])+1)

#rotate and mirror p
def manip(p):
    z=[]
    #print(p)
    #render_piece(p)
    z.append((( 1, 0),( 0, 1)))
    z.append((( 0, 1),(-1, 0)))
    z.append(((-1, 0),( 0,-1)))
    z.append((( 0,-1),( 1, 0)))
    
    z.append((( 0, 1),( 1, 0)))
    z.append(((-1, 0),( 0, 1)))
    z.append((( 0,-1),(-1, 0)))
    z.append((( 1, 0),( 0,-1)))
    
    r=[]
    l=len(z)
    for i in range(l):
        (dxx,dxy),(dyx,dyy)=z[i]
        r.append(home([(x*dxx+y*dxy,x*dyx+y*dyy) for x,y in p]))
        #render_piece(r[-1])
    return r
  
def col(x):
    return [100+(x*13)%156,(x*27)%256,(x*17)%256]
  

def render_map(M,w,h):
    pix=np.zeros([h,w,3]).astype(np.ubyte)
    
    for x,y in M.keys():
        pix[y,x,:]=M[(x,y)]
    F=16
    pix = cv2.resize(pix, fx=F, fy=F, dsize=(0, 0), interpolation=cv2.INTER_NEAREST)
    #display.clear_output(wait=True)
    display.display(Image.fromarray(pix, 'RGB'))

def render_piece(pts):
    M={}
    for p in pts:
        M[p]=[255,0,255]
    
    w,h=sz(M)
    render_map(M,h,w)
    
        
    
    
def solve1(fn):
    settings, pieces=load(fn)
    #print("settings:\n",settings)
    #print("pieces:\n",pieces)
    def check(w,h,indices,M):
        #print(w,h,indices)
        if len(indices)==0:
            render_map(M,w,h)
            print("FIT")
            return 1
        
        #render_map(M,w,h)
        nonlocal pieces
        #for all piece orientations of first piece
        for mpp in manip(pieces[ indices[0] ]):
            pw,ph=sz(mpp)
            #print(mpp,pw,ph)
            #for all possible locations
            for x in range(w-pw+1):
                for y in range(h-ph+1):
                    #try fit
                    fit=1
                    MM=dict(M)
                    for xx,yy in mpp:
                        pp=(x+xx,y+yy)
                        if pp in MM:
                            fit=0
                            break
                        MM[pp]=col(len(indices))
                    if fit:
                        #render_map(MM,w,h)
                        #place next
                        if check(w,h,indices[1:],MM):
                            return 1
        return 0
    
    r=0
    for w,h,indices in settings:
        print("checking:",w,h,indices[:10])
        
        #do other checks
        
        v=check(w,h,indices,{})
        print("result:",v)
        r+=v
    return(r)   
#print("test1:",solve1("12.tst"),2)
#print("part1:",solve1("12.txt"),0)
    
    

In [2]:
#I read a comment that a simple volume check worked
#a bit frustrated as i spent quite some time
#building a solution that explodes in runtime.

#feels a bit like chemistry

#merry christmas whoever you are


def solve1(fn):
    settings, pieces=load(fn)
    #print("settings:\n",settings)
    #print("pieces:\n",pieces)
    def check(w,h,indices,M):
        pz=0
        nonlocal pieces
        for i in indices:
            pz+=len(pieces[i])
        return 0 if pz>w*h else 1
    
    r=0
    for w,h,indices in settings:
        #print("checking:",w,h,indices)
        
        #do other checks
        
        v=check(w,h,indices,{})
        #print("result:",v)
        r+=v
    return(r)   
print("test1:",solve1("12.tst"),2)
print("part1:",solve1("12.txt"),587)
    




test1: 3 2
part1: 587 587
